# Synova RAG - Generate Embeddings

## Objective

This notebook converts the multilingual knowledge base into dense vector embeddings.

The workflow includes:

1. Load processed RAG documents.
2. Load the multilingual embedding model.
3. Generate embeddings.
4. Build a FAISS vector index.
5. Save the vector database for inference.

# 02 Import Libraries

In [1]:
import pandas as pd
from pathlib import Path

import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

# 03 Paths

In [3]:
PROJECT_ROOT = Path.cwd().parent

VECTOR_DB_DIR = PROJECT_ROOT / "vector_db"

# 04 Load RAG Documents

In [4]:
rag_df = pd.read_csv(
    VECTOR_DB_DIR / "rag_documents.csv"
)

rag_df.head()

,id,title,language,category,section,text
0,tech_password_access,إعادة تعيين كلمة المرور والوصول إلى الحساب,ar,technical,نظرة عامة,تُعد مشكلات تسجيل الدخول والوصول إلى الحساب من...
1,tech_password_access,إعادة تعيين كلمة المرور والوصول إلى الحساب,ar,technical,المشكلات الشائعة,نسيان كلمة المرور\nقفل الحساب\nاسم مستخدم أو ك...
2,tech_password_access,إعادة تعيين كلمة المرور والوصول إلى الحساب,ar,technical,الأعراض,عدم القدرة على تسجيل الدخول\nظهور رسالة تفيد ب...
3,tech_password_access,إعادة تعيين كلمة المرور والوصول إلى الحساب,ar,technical,الأسباب المحتملة,إدخال كلمة مرور غير صحيحة\nانتهاء صلاحية كلمة ...
4,tech_password_access,إعادة تعيين كلمة المرور والوصول إلى الحساب,ar,technical,خطوات استكشاف المشكلة,تأكد من صحة اسم المستخدم.\nأعد تعيين كلمة المر...


# 05 Number of Chunks

In [5]:
print(len(rag_df))

28


# 06 Load Embedding Model

In [6]:
MODEL_NAME = "BAAI/bge-m3"

embedding_model = SentenceTransformer(MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\balke\Desktop\AI SPIRE\Capstone Project\COPY\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\balke\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

# 07 Generate Embeddings

In [7]:
embeddings = embedding_model.encode(
    rag_df["text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

# 08 Check Shape

In [8]:
embeddings.shape

(28, 1024)

# 09 Build FAISS Index

In [9]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

# 10 Number of Vectors

In [10]:
index.ntotal

28

# 11 Save Index

In [11]:
faiss.write_index(
    index,
    str(VECTOR_DB_DIR / "synova.index")
)

# 12 Save Metadata

In [12]:
rag_df.to_pickle(
    VECTOR_DB_DIR / "metadata.pkl"
)

# 13 Final Check

In [13]:
print("Embedding Shape :", embeddings.shape)
print("Vectors Stored  :", index.ntotal)

Embedding Shape : (28, 1024)
Vectors Stored  : 28


In [14]:
import json

config = {
    "embedding_model": MODEL_NAME,
    "dimension": embeddings.shape[1],
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "knowledge_base_version": "1.0"
}

with open(VECTOR_DB_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)